# Practical 7: Sentiment Analysis with Bidirectional LSTM

**Aim:** Classify IMDB movie reviews as positive or negative using an Embedding layer and stacked Bidirectional LSTMs.

In [ ]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Embedding,Bidirectional,LSTM,Dense,Dropout,BatchNormalization

from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("IMDB-Dataset.csv", engine='python', on_bad_lines='skip')

In [ ]:
df.head()

In [ ]:
sentences = df['review']
labels = df['sentiment'].map({'negative': 0, 'positive': 1})

In [ ]:
labels

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(np.array(sentences),
np.array(labels), train_size=0.8, random_state=42)

In [ ]:
# X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, train_size=0.5, random_state=42)

In [ ]:
print(f'X train Shape : {X_train.shape}')
print(f'y train Shape : {y_train.shape}')
print()
# print(f'X val Shape : {X_val.shape}')
# print(f'y val Shape : {y_val.shape}')
# print()
print(f'X test Shape : {X_test.shape}')
print(f'y test Shape : {y_test.shape}')

In [ ]:
total_words = 1000
max_length = 120

In [ ]:
tokenizer = Tokenizer(num_words=total_words,oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

In [ ]:
len(tokenizer.word_index)

In [ ]:
train_seq = tokenizer.texts_to_sequences(X_train)
train_padd = pad_sequences(train_seq,maxlen=max_length,truncating='post')

In [ ]:
# val_seq = tokenizer.texts_to_sequences(X_val)
# val_padd = pad_sequences(val_seq,maxlen=max_length,truncating='post')

In [ ]:
test_seq = tokenizer.texts_to_sequences(X_test)
test_padd = pad_sequences(test_seq,maxlen=max_length,truncating='post')

In [ ]:
max_seq_len = max([len(x) for x in train_seq])

In [ ]:
# example on train
index = 5
print(f'Text :\n{X_train[index]}\n\n')
print(f'Sequence :\n{train_seq[index]}\n\n')
print(f'Padded Sequence :\n{train_padd[index]}\n\n')

In [ ]:
# # example on val
# index = 5
# print(f'Text :\n{X_val[index]}\n\n')
# print(f'Sequence :\n{val_seq[index]}\n\n')
# print(f'Padded Sequence :\n{val_padd[index]}\n\n')

In [ ]:
# # example on val
# index = 5
# print(f'Text :\n{X_val[index]}\n\n')
# print(f'Sequence :\n{val_seq[index]}\n\n')
# print(f'Padded Sequence :\n{val_padd[index]}\n\n')

In [ ]:
embedding_dim = 16
model = Sequential([
        Embedding(total_words,embedding_dim,),
        Bidirectional(LSTM(64,return_sequences=True)),
        Bidirectional(LSTM(32)),
        Dense(6,activation='relu'),
        Dense(1,activation='sigmoid')
])

In [ ]:
model.summary()

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

callbacks = [
    EarlyStopping(patience=10, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=0.000001, verbose=1),
    ModelCheckpoint('model.keras', monitor='val_loss', save_best_only=True, verbose=1)
]

In [ ]:
model.compile(optimizer=Adam(0.001),loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
history=model.fit(train_padd,y_train,epochs=20,validation_data=(test_padd,y_test),callbacks=[callbacks])

In [ ]:
def plot_graphs(history, string):
        plt.plot(history.history[string])
        plt.plot(history.history['val_'+string])
        plt.xlabel("Epochs")
        plt.ylabel(string)
        plt.legend([string, 'val_'+string])
        plt.show()

plot_graphs(history, 'accuracy')
plot_graphs(history, 'loss')